# PharmaGuard: evaluate on a public dataset (data kept in Google Drive)

Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom.

- The dataset zip and all results are stored in **Google Drive** (`MyDrive/PharmaGuard/`), so they survive Colab restarts.
- Videos are unzipped to the Colab disk for speed (reading video straight from Drive is slow). Drive keeps the zip.
- Nothing is trained. PharmaGuard uses pretrained YOLOv8-pose plus behaviour rules; this notebook **measures** them.

In [ ]:
import torch, platform
print("Python", platform.python_version(), "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/PharmaGuard'
for sub in ('datasets', 'configs', 'results'):
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)
print('Drive folder ready:', DRIVE_DIR)

## 2. Get the code

In [ ]:
BRANCH = 'fyp-evaluation-inventory'   # change to 'main' after merging
!rm -rf /content/pharmguard
!git clone -q --depth 1 -b {BRANCH} https://github.com/Franklinnn099/Pharmguard.git /content/pharmguard
%cd /content/pharmguard
!pip install -q gradio==6.32.0 ultralytics==8.4.215 "pydantic>=2.5,<3" scipy
!git log --oneline -1

## 3. Put the dataset in Drive

**Option A (link):** on the Mendeley dataset page (e.g. *Shoplifting Dataset (2022), MNNIT*, DOI 10.17632/r3yjf35hzr.1),
press **Download All**. Copy the download link from your browser's downloads list and paste it below.
Colab downloads it straight into Drive.

**Option B (upload):** download the zip on your laptop and upload it into `MyDrive/PharmaGuard/datasets/` with the
Drive app. Leave `DOWNLOAD_URL` empty.

In [ ]:
DATASET = 'mnnit'          # short name, used for folders and result files
DOWNLOAD_URL = ''          # Option A: paste the link here. Option B: leave empty
ZIP_IN_DRIVE = f'{DRIVE_DIR}/datasets/{DATASET}.zip'

if DOWNLOAD_URL and not os.path.exists(ZIP_IN_DRIVE):
    !wget -q --show-progress -O "{ZIP_IN_DRIVE}" "{DOWNLOAD_URL}"
if not os.path.exists(ZIP_IN_DRIVE):
    zips = [f for f in os.listdir(f'{DRIVE_DIR}/datasets') if f.lower().endswith('.zip')]
    print('Zip files in Drive:', zips)
    raise SystemExit(f'Put the dataset zip at {ZIP_IN_DRIVE} (or rename yours to {DATASET}.zip)')
print('Zip in Drive:', ZIP_IN_DRIVE, f'({os.path.getsize(ZIP_IN_DRIVE)/1e6:.0f} MB)')

In [ ]:
LOCAL = f'/content/clips/{DATASET}'
!rm -rf "{LOCAL}" && mkdir -p "{LOCAL}"
!unzip -q "{ZIP_IN_DRIVE}" -d "{LOCAL}"
# Some zips contain zips (one per class): unpack those too.
!find "{LOCAL}" -name "*.zip" -execdir unzip -q -o {} \;
VIDEO_EXT = ('.mp4', '.avi', '.mov', '.mkv', '.mpg', '.mpeg', '.webm')
from collections import Counter
folders = Counter()
for d, _, files in os.walk(LOCAL):
    n = sum(f.lower().endswith(VIDEO_EXT) for f in files)
    if n:
        folders[os.path.relpath(d, LOCAL)] += n
for f, n in sorted(folders.items()):
    print(f'{n:5d} videos  in  {f}')

## 4. Point the labels at the class folders

`DATA_ROOT` must be the folder that directly contains the class folders (e.g. `Normal/` and `Shoplifting/`).
Look at the listing above and set it, then map each class folder to a label.

In [ ]:
DATA_ROOT = LOCAL          # e.g. f'{LOCAL}/Shoplifting Dataset' if the zip has an outer folder
LABEL_MAP = 'shoplifting=alert normal=none'   # folder=label; folder names are case-insensitive
print(sorted(e for e in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, e))))

## 5. Draw the shelf zone for this camera view

Without the right zone, no hand ever "touches the shelf" and every clip scores *none*.
The frame below has a 10% grid. Edit `ZONE` (x, y as fractions: 0,0 = top-left, 1,1 = bottom-right) around the
shelf front where hands reach in, and re-run until the green outline sits on the shelf.
If clips were filmed from different angles, check a few clips and split them by angle (one zone per angle).

In [ ]:
import cv2, glob, numpy as np, matplotlib.pyplot as plt
clips = sorted(p for p in glob.glob(f'{DATA_ROOT}/**/*', recursive=True) if p.lower().endswith(VIDEO_EXT))
SAMPLE = clips[0]          # try a few: clips[10], clips[-1] ...
ZONE = [[0.15, 0.10], [0.85, 0.10], [0.85, 0.55], [0.15, 0.55]]

cap = cv2.VideoCapture(SAMPLE); cap.set(cv2.CAP_PROP_POS_FRAMES, 15); ok, frame = cap.read(); cap.release()
img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
for i in range(1, 10):
    cv2.line(img, (int(w*i/10), 0), (int(w*i/10), h), (200, 200, 200), 1)
    cv2.line(img, (0, int(h*i/10)), (w, int(h*i/10)), (200, 200, 200), 1)
cv2.polylines(img, [np.array([[x*w, y*h] for x, y in ZONE], np.int32)], True, (0, 255, 0), 3)
plt.figure(figsize=(10, 7)); plt.imshow(img); plt.title(os.path.relpath(SAMPLE, DATA_ROOT)); plt.axis('off'); plt.show()

In [ ]:
import yaml
CONFIG_IN_DRIVE = f'{DRIVE_DIR}/configs/{DATASET}_view.yaml'
cfg = {'zones': [{'name': 'Shelf', 'polygon_normalized': ZONE}]}
with open(CONFIG_IN_DRIVE, 'w') as f:
    yaml.safe_dump(cfg, f)
print('Saved zone config to Drive:', CONFIG_IN_DRIVE)

## 6. Check one clip first

Writes an annotated video so you can SEE whether wrists are found and the zone lights up.
If wrists are missing or the zone never shows HAND CONTACT, fix the zone before scoring everything.

In [ ]:
!python tools/track_video.py "{SAMPLE}" --fps 5 --config "{CONFIG_IN_DRIVE}" --out /content/sample_annotated.mp4
!cp /content/sample_annotated.mp4 "{DRIVE_DIR}/results/{DATASET}_sample_annotated.mp4"
print('Open MyDrive/PharmaGuard/results/ in Drive to watch it.')

## 7. Score every clip

Results go to Drive. Takes roughly the clip length / speed-up shown by the tool; on a T4 expect several clips per minute.
`MAX_CLIPS` lets you run a quick subset first.

In [ ]:
import csv, random
!python tools/make_labels.py "{DATA_ROOT}" --map {LABEL_MAP} --config "{CONFIG_IN_DRIVE}" --out /content/labels_all.csv
MAX_CLIPS = 60     # None = all clips
rows = list(csv.DictReader(open('/content/labels_all.csv')))
for r in rows:     # make clip paths absolute so the labels file can live anywhere
    r['clip'] = os.path.join(DATA_ROOT, r['clip'])
random.seed(0)
by = {}
for r in rows:
    by.setdefault(r['expected'], []).append(r)
if MAX_CLIPS:
    per = max(1, MAX_CLIPS // len(by))
    rows = [r for v in by.values() for r in random.sample(v, min(per, len(v)))]
with open('/content/labels.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['clip', 'scenario', 'expected', 'notes', 'config']); w.writeheader(); w.writerows(rows)
print(len(rows), 'clips selected', Counter(r['expected'] for r in rows))

In [ ]:
RESULTS = f'{DRIVE_DIR}/results/{DATASET}_results.csv'
!python tools/evaluate_clips.py /content/labels.csv --fps 5 --out "{RESULTS}" | tee "{DRIVE_DIR}/results/{DATASET}_summary.txt"

## 8. What to do with the numbers

- Paste `*_summary.txt` into the report; attach `*_results.csv` as an appendix.
- Open 3-5 WRONG clips with `tools/track_video.py ... --out` (cell 6) to see why they failed: zone, camera angle,
  hidden hands, or a rule. Report the reasons, do not hide the clips.
- Thresholds (`configs/demo_shelf.yaml`: `carry_window_sec`, `pocket_radius_torso`) may be adjusted on HALF the clips,
  then the other half reported. Tuning and reporting on the same clips inflates the result.
- State clearly: a public staged retail dataset, not a pharmacy and not the demo camera placement.